# Bias Analysis — Credit Approval Decisions

This notebook evaluates potential bias in historical credit approval outcomes, focusing on **gender-based disparate impact (DI)** and **proxy discrimination**.

## 1. Data & Setup

We run the analysis on the cleaned / analysis-ready dataset produced by the data-quality pipeline.  
We report sample size and confirm the variables required for bias analysis.

When check all feature columns and it will support a foundation to do DI analysis.

There are not age, brith date or other features so following bias analysis regard gender.

In [17]:
import sys
from pathlib import Path
import importlib
import pandas as pd
from scipy.stats import chi2_contingency

sys.path.append(str(Path("..").resolve()))

import src.data_clean_notebook as preproc
importlib.reload(preproc)

df = preproc.run_data_quality_pipeline("../data/raw_credit_applications.json")

Final dataset shape: (502, 27)


In [18]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())

shape: (502, 27)
columns: ['applicant_gender', 'applicant_zip_code', 'fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'decision_loan_approved', 'spend_shopping', 'spend_rent', 'spend_alcohol', 'spend_txn_count', 'spend_total', 'spend_mean', 'spend_max', 'spend_unique_cats', 'spend_dining', 'spend_healthcare', 'spend_fitness', 'spend_entertainment', 'spend_insurance', 'spend_travel', 'spend_transportation', 'spend_utilities', 'spend_groceries', 'spend_education', 'spend_adult_entertainment', 'spend_gambling']


In [19]:
# Minimal sanity checks
print("\nRaw gender counts:")
print(df["applicant_gender"].value_counts(dropna=False))

print("\nRaw label counts:")
print(df["decision_loan_approved"].value_counts(dropna=False))


Raw gender counts:
applicant_gender
Male      195
Female    193
F          58
M          53
NaN         3
Name: count, dtype: int64

Raw label counts:
decision_loan_approved
True     292
False    210
Name: count, dtype: int64


## 2. Definitions

- **Outcome:** loan approval label `y` (1 = approved, 0 = rejected) derived from `decision_loan_approved`.
- **Protected attribute:** gender derived from `applicant_gender` and normalized into `gender_norm` (male/female).
- **Selection rate:** group approval rate, used for DI computation.

Fairness metrics are computed only on rows where both the protected attribute and outcome are defined.

Rows with missing or unmapped gender are excluded from DI because DI is undefined without group membership.

In [20]:
# If y exists, compare approval rate
df["y"] = df["decision_loan_approved"].astype(int)

## 3. Missingness & Validity Checks (Protected Attribute)

Before computing fairness metrics, we audit missingness in the protected attribute.  
If missingness correlates with outcomes, excluding missing rows may distort fairness estimates.

We report:
- missing rate for `applicant_gender`
- outcome distribution and approval rate for missing vs non-missing rows

In [21]:
# Missingness audit on raw applicant_gender

raw_gender = df["applicant_gender"]
miss_raw_gender = raw_gender.isna() | (raw_gender.astype(str).str.strip() == "")

print(f"Missing applicant_gender rate: {miss_raw_gender.mean():.4%}")
print("Counts (missing vs non-missing):")
print(miss_raw_gender.value_counts())


print("\nApproval rate when raw gender is missing:", df.loc[miss_raw_gender, "y"].mean() if miss_raw_gender.any() else "N/A")
print("Approval rate when raw gender is NOT missing:", df.loc[~miss_raw_gender, "y"].mean())

Missing applicant_gender rate: 0.5976%
Counts (missing vs non-missing):
applicant_gender
False    499
True       3
Name: count, dtype: int64

Approval rate when raw gender is missing: 0.6666666666666666
Approval rate when raw gender is NOT missing: 0.5811623246492986


**Missingness audit**

The missing rate for applicant_gender is 0.60% (3/502), which is minimal. The approval rate for missing-gender rows is 0.667 (2/3) versus 0.581 for non-missing rows. Because the missing group is extremely small, excluding these rows is unlikely to materially change DI; however, we report this tr


## 4. Group Descriptives (Gender)

We report:
- group sample sizes
- selection/approval rates by gender

These descriptives provide context for DI and help interpret whether disparities are driven by base-rate differences.

We first compute approval/selection rates by group and then compute Disparate Impact (DI). DI is computed on rows with non-missing protected attribute and outcome, because DI is undefined otherwise. 

In [22]:
# Create gender_norm WITHOUT hiding missing values

g = df["applicant_gender"].astype("string").str.strip().str.lower()

gender_map = {
    "m": "male", "male": "male", "man": "male",
    "f": "female", "female": "female", "woman": "female",
}

df["gender_norm"] = g.map(gender_map)

# IMPORTANT: keep missing as <NA> (do NOT fillna here)
# df["gender_norm"] remains NA if raw gender was missing or unrecognized

# (optional) show what values are not mapped
print("gender_norm value counts (including NA):")
print(df["gender_norm"].value_counts(dropna=False))

gender_norm value counts (including NA):
gender_norm
female    251
male      248
NaN         3
Name: count, dtype: int64


## 5. Disparate Impact (DI) by Gender







### 5.1 Method
DI is computed as:

\[
DI = \frac{\text{selection rate of unprivileged group}}{\text{selection rate of privileged group}}
\]

We use the common convention:
- privileged = **male**
- unprivileged = **female**

We compute DI only on rows where both `gender_norm` and `y` are non-missing (DI is undefined otherwise).

In [23]:
# Prepare DI dataset (DI is undefined when gender is missing)

tmp = df[["gender_norm", "y"]].dropna().copy()
tmp["y"] = tmp["y"].astype(int)

print("DI dataset size:", tmp.shape[0])
print(tmp["gender_norm"].value_counts())

group_sizes = tmp["gender_norm"].value_counts()
approval_rates = tmp.groupby("gender_norm")["y"].mean().sort_values(ascending=False)

print("Group sizes:\n", group_sizes)
print("\nApproval rates:\n", approval_rates)

DI dataset size: 499
gender_norm
female    251
male      248
Name: count, dtype: int64
Group sizes:
 gender_norm
female    251
male      248
Name: count, dtype: int64

Approval rates:
 gender_norm
male      0.657258
female    0.505976
Name: y, dtype: float64


### 5.2 Computation (selection rates + DI)
We compute group sizes and selection rates, then calculate DI.

In [24]:
priv_group = "male"
unpriv_group = "female"

if priv_group in approval_rates.index and unpriv_group in approval_rates.index:
    di = float(approval_rates.loc[unpriv_group] / approval_rates.loc[priv_group])
    print(f"DI (female/male) = {di:.4f}")
    print(f"80% rule flag (DI < 0.8): {di < 0.8}")
else:
    print("Cannot compute DI: missing male or female group in the DI dataset.")
    print("Available groups:", approval_rates.index.tolist())

DI (female/male) = 0.7698
80% rule flag (DI < 0.8): True


### 5.3 Interpretation (Four-Fifths Rule)
We interpret DI using the four-fifths rule:
- **DI < 0.80** triggers a screening flag for potential adverse impact.
DI is descriptive and does not establish causality; therefore we complement it with proxy checks.

Conclusion — Gender DI (Disparate Impact)

Missingness check: applicant_gender is missing for 3/502 (0.60%) records. The approval rate for the missing-gender rows is 0.667 (2/3) vs 0.581 for non-missing rows. Because the missing group is extremely small, excluding these rows is unlikely to materially change the DI estimate, but we report it transparently.

Group selection rates: On the DI computation subset (n=499 with non-missing gender), the approval/selection rate is 0.657 for male (n=248) and 0.506 for female (n=251).

Disparate Impact (DI): Using the common convention DI = female_rate / male_rate, we obtain DI = 0.7698. This falls below the 0.80 four-fifths threshold, so the 80% rule is triggered, indicating potential adverse impact against the female group in the observed decision outcomes.

Note on interpretation: DI is a screening metric, not a causal proof of discrimination. Next steps include investigating possible proxy drivers (e.g., zip code or financial variables correlated with gender) and checking whether the disparity persists after controlling for relevant risk factors.

## 6. Statistical Association (Gender vs Approval)

To complement DI, we test whether approval outcomes are associated with gender using:
- **Chi-square test of independence** on the contingency table (gender × approval)
- **Cramer’s V** as an effect size (strength of association)

We test whether approval decisions are independent of gender.

H0: gender and approval are independent.

If p < 0.05, we reject H0 and conclude the difference is statistically significant.
This helps distinguish “numerical disparity” from “statistically supported association”.

### Chi-square test of independence

In [25]:


ct = pd.crosstab(tmp["gender_norm"], tmp["y"])
print("Contingency table (gender x approval):\n", ct)

chi2, p, dof, expected = chi2_contingency(ct)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
print("\nExpected counts under H0 (independence):\n", expected_df.round(2))

Contingency table (gender x approval):
 y              0    1
gender_norm          
female       124  127
male          85  163

Chi-square statistic: 11.1156
p-value: 0.000856
Degrees of freedom: 1

Expected counts under H0 (independence):
 y                 0       1
gender_norm                
female       105.13  145.87
male         103.87  144.13


**Chi-Square conclusion**

The chi-square test indicates that loan approval decisions are not independent of gender (χ²(1)=11.12, p=0.000856). Therefore, we reject H0 and conclude that the difference in approval outcomes across gender groups is statistically significant. The observed pattern is consistent with the fairness metrics above: the approval rate is higher for Male than for Female, supporting the presence of disparate outcomes that warrant further investigation.

### Cramer's V test
We report Cramer’s V to quantify the strength of association between gender and approval.

In [26]:
import numpy as np

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))
print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.1493


**Cramer's V Conclusion**

The effect size (Cramer’s V = 0.1493) suggests the gender–approval relationship is weak-to-moderate. This supports monitoring and audit actions: even modest associations can be problematic in decision systems, especially when DI violates the four-fifths rule.

## 7. Robustness Checks

We run small checks to ensure the DI result is not an artifact of:
- gender normalization rules
- dropping a very small number of missing protected-attribute rows
- inclusion/exclusion of non-binary/unknown gender values

In [27]:
# DI computed on non-missing gender
baseline_n = tmp.shape[0]
print("Baseline DI dataset size:", baseline_n)

# If there are very few missing gender rows, report it explicitly
missing_gender_n = df["gender_norm"].isna().sum()
print("Rows with missing/unmapped gender_norm:", int(missing_gender_n))

Baseline DI dataset size: 499
Rows with missing/unmapped gender_norm: 3


**Conclusion (Robustness)**  
Only 3/502 rows have missing/unmapped gender, so the DI estimate is not driven by dropping a large number of protected-attribute rows.

## 8. Proxy Discrimination Analysis

Even if gender is not explicitly used, non-protected variables may indirectly encode gender information and contribute to disparities.  
We assess proxy discrimination using a two-link logic:

1) **Proxy carries gender information** (feature ↔ gender)  
2) **Proxy influences decisions** (feature ↔ approval outcome)

A stronger proxy risk exists when **both links** are supported by evidence.

### 8.1 Sanity check

We create:
- `y`: binary approval outcome (1 = approved, 0 = rejected)
- `gender_norm`: normalized gender label (male/female).  
Missing or unmapped values are kept as missing.

In [ ]:
# 8.1 Sanity check: required variables exist and look reasonable

required = ["y", "gender_norm", "applicant_zip_code"]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for proxy analysis: {missing_cols}")

print("y value counts:")
print(df["y"].value_counts(dropna=False))

print("\ngender_norm value counts (incl. NA):")
print(df["gender_norm"].value_counts(dropna=False))

print("\nZIP code missing rate:", df["applicant_zip_code"].isna().mean())

gender_norm counts (incl. NA):
gender_norm
female    251
male      248
NaN         3
Name: count, dtype: int64

y counts:
y
1    292
0    210
Name: count, dtype: int64


**Conclusion (8.1 — Sanity check)**  
The analysis variables are well-defined: `gender_norm` contains **female=251**, **male=248**, and **missing=3**, while the outcome `y` contains **approved=292** and **rejected=210**.  
Therefore, proxy tests can be conducted on consistent definitions of protected attribute and outcome.

### 8.2 Proxy link (1): ZIP code ↔ Gender

We test whether ZIP code is statistically associated with gender.  
If ZIP strongly correlates with gender, it indicates **proxy potential** (demographic encoding).  
We report χ², p-value, Cramér’s V, and a sparsity check on expected counts.

In [32]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

zip_gender = df[["applicant_zip_code", "gender_norm"]].dropna().copy()
zip_gender["applicant_zip_code"] = zip_gender["applicant_zip_code"].astype(str)

ct_zg = pd.crosstab(zip_gender["applicant_zip_code"], zip_gender["gender_norm"])
print("Contingency table shape (ZIP x gender):", ct_zg.shape)

chi2_zg, p_zg, dof_zg, exp_zg = chi2_contingency(ct_zg)

n = ct_zg.to_numpy().sum()
r, k = ct_zg.shape
cramers_v_zg = np.sqrt(chi2_zg / (n * (min(r - 1, k - 1))))

print(f"Chi-square (ZIP vs gender): {chi2_zg:.4f}")
print(f"p-value: {p_zg:.6f}")
print(f"Cramer's V: {cramers_v_zg:.4f}")

# Sparsity diagnostics (χ² reliability indicator)
print(f"Min expected count: {exp_zg.min():.2f}")
print(f"Share of expected cells < 5: {(exp_zg < 5).mean():.2%}")

Contingency table shape (ZIP x gender): (195, 2)
Chi-square (ZIP vs gender): 393.7343
p-value: 0.000000
Cramer's V: 0.8883
Min expected count: 0.50
Share of expected cells < 5: 100.00%


**Conclusion (8.2 — ZIP code ↔ Gender)**  
The χ² test indicates a statistically significant association between ZIP code and gender (p < 0.001). However, the contingency table is **highly sparse**: the minimum expected count is **0.50** and **100%** of expected cells are below 5.  
This violates the usual χ² assumptions, so the p-value and Cramér’s V should be treated as **indicative rather than definitive**.  
Nevertheless, the result suggests that ZIP code likely encodes demographic information and is a reasonable **proxy candidate** to evaluate further.

### 8.3 Proxy link (2): ZIP code ↔ Approval outcome

Because ZIP has many categories and produces sparse contingency tables, we evaluate the ZIP–approval link using aggregated ZIP groups with sufficient support.  
We report approval-rate differences across ZIP groups and test association after grouping.

In [34]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# Build ZIP stats
zip_out = df[["applicant_zip_code", "y"]].dropna().copy()
zip_out["applicant_zip_code"] = zip_out["applicant_zip_code"].astype(str)
zip_out["y"] = zip_out["y"].astype(int)

zip_counts = zip_out["applicant_zip_code"].value_counts()
top_k = 20  # you can adjust (e.g., 10/20/30)
top_zips = set(zip_counts.head(top_k).index)

zip_out["zip_group"] = np.where(zip_out["applicant_zip_code"].isin(top_zips),
                                zip_out["applicant_zip_code"],
                                "OTHER")

# Approval rates by grouped ZIP
group_stats = zip_out.groupby("zip_group")["y"].agg(n="count", approval_rate="mean").sort_values("n", ascending=False)
display(group_stats.head(25))

# Chi-square on grouped table (less sparse)
ct = pd.crosstab(zip_out["zip_group"], zip_out["y"])
chi2, p, dof, exp = chi2_contingency(ct)

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

print("Grouped contingency table shape (zip_group x y):", ct.shape)
print(f"Chi-square (grouped ZIP vs approval): {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Cramer's V: {cramers_v:.4f}")
print(f"Min expected count: {exp.min():.2f}")
print(f"Share of expected cells < 5: {(exp < 5).mean():.2%}")

,n,approval_rate
zip_group,,
OTHER,389,0.570694
10048,8,0.750000
90284,7,0.428571
10096,7,0.285714
10004,6,1.000000
10057,6,0.500000
10020,6,0.666667
10019,6,0.333333
10002,5,0.600000


Grouped contingency table shape (zip_group x y): (21, 2)
Chi-square (grouped ZIP vs approval): 25.9893
p-value: 0.166167
Cramer's V: 0.2280
Min expected count: 2.09
Share of expected cells < 5: 95.24%


**Conclusion (8.3 — ZIP code ↔ Approval, grouped ZIPs)**  
After grouping ZIP codes into the **top 20 most frequent ZIPs + OTHER**, the association between ZIP group and approval outcome is **not statistically significant** (χ² p = **0.166**).  
The estimated effect size is **Cramér’s V = 0.228** (small-to-moderate). However, the grouped contingency table still shows **substantial sparsity** (min expected = **2.09**, and **95.24%** of expected cells are < 5), so results should be interpreted cautiously.  
Overall, we do not find strong evidence that ZIP code is a direct driver of approval outcomes in this dataset.

### 8.5 Financial proxy candidate: Annual income (`fin_annual_income`)

ZIP codes are highly sparse, so we also evaluate a financial variable that is more plausibly tied to approval decisions.  
We apply the same two-link proxy logic:

1) Income ↔ Gender (proxy potential)  
2) Income ↔ Approval (decision relevance)

In [35]:
# 8.5.1 Income ↔ Gender (group differences)

income_gender = df[["gender_norm", "fin_annual_income"]].dropna().copy()
income_gender = income_gender[income_gender["gender_norm"].isin(["male", "female"])]

summary_income = income_gender.groupby("gender_norm")["fin_annual_income"].agg(
    n="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
)

display(summary_income)

# Simple standardized mean difference (SMD) as effect size
male_income = income_gender.loc[income_gender["gender_norm"] == "male", "fin_annual_income"]
female_income = income_gender.loc[income_gender["gender_norm"] == "female", "fin_annual_income"]

pooled_std = np.sqrt((male_income.var(ddof=1) + female_income.var(ddof=1)) / 2)
smd = (female_income.mean() - male_income.mean()) / pooled_std if pooled_std > 0 else np.nan

print(f"SMD (female - male) for income: {smd:.3f}")

,n,mean,median,std,min,max
gender_norm,,,,,,
female,246,84253.791826,83000.0,28304.842591,22000.0,171000.0
male,247,81388.663968,81000.0,27486.632986,22000.0,168000.0


SMD (female - male) for income: 0.103


In [36]:
# 8.5.2 Income ↔ Approval (approval rate by income quantiles)

income_out = df[["y", "fin_annual_income"]].dropna().copy()
income_out["y"] = income_out["y"].astype(int)

# 5 quantile bins (adjustable)
income_out["income_bin"] = pd.qcut(income_out["fin_annual_income"], q=5, duplicates="drop")

bin_stats = income_out.groupby("income_bin")["y"].agg(n="count", approval_rate="mean")
display(bin_stats)

# Optional: overall correlation check (not causal, just directional)
corr = income_out[["fin_annual_income", "y"]].corr().iloc[0, 1]
print(f"Correlation(income, approval y): {corr:.3f}")

/var/folders/6m/9yqj6yd90d10n4czxh4lntpc0000gn/T/ipykernel_12409/3444859089.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bin_stats = income_out.groupby("income_bin")["y"].agg(n="count", approval_rate="mean")


,n,approval_rate
income_bin,,
"(21999.999, 58000.0]",101,0.396040
"(58000.0, 75000.0]",101,0.544554
"(75000.0, 89000.0]",99,0.666667
"(89000.0, 105000.0]",97,0.649485
"(105000.0, 171000.0]",98,0.663265


Correlation(income, approval y): 0.178


**Conclusion (8.5 — Annual income as a proxy candidate)**  

**Link (1): Income ↔ Gender.** Female and male income distributions are very similar (female mean ≈ 84,254 vs male mean ≈ 81,389; medians 83,000 vs 81,000). The standardized mean difference is **SMD = 0.103**, indicating only a **small** gender difference in income. This suggests **limited proxy potential** for gender via income.

**Link (2): Income ↔ Approval.** Approval rates increase substantially across income quantiles (from **0.396** in the lowest bin to roughly **0.66** in higher bins). The correlation between income and approval is positive (**r = 0.178**), indicating income is **decision-relevant**.

**Overall:** Income appears to be a legitimate risk-related driver of approval, but based on the small gender difference (SMD ≈ 0.10), it is **not a strong gender proxy** in this dataset.

### 8.6 Financial proxy candidate: Debt-to-income ratio (`fin_debt_to_income`)

We repeat the same two-link proxy logic for DTI:
1) DTI ↔ Gender (proxy potential)  
2) DTI ↔ Approval (decision relevance)

In [37]:
# 8.6.1 DTI ↔ Gender (group differences)

dti_gender = df[["gender_norm", "fin_debt_to_income"]].dropna().copy()
dti_gender = dti_gender[dti_gender["gender_norm"].isin(["male", "female"])]

summary_dti = dti_gender.groupby("gender_norm")["fin_debt_to_income"].agg(
    n="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
)
display(summary_dti)

male_dti = dti_gender.loc[dti_gender["gender_norm"] == "male", "fin_debt_to_income"]
female_dti = dti_gender.loc[dti_gender["gender_norm"] == "female", "fin_debt_to_income"]

pooled_std = np.sqrt((male_dti.var(ddof=1) + female_dti.var(ddof=1)) / 2)
smd_dti = (female_dti.mean() - male_dti.mean()) / pooled_std if pooled_std > 0 else np.nan
print(f"SMD (female - male) for DTI: {smd_dti:.3f}")

,n,mean,median,std,min,max
gender_norm,,,,,,
female,251,0.243865,0.22,0.154939,0.05,1.85
male,248,0.248952,0.24,0.114381,0.05,0.45


SMD (female - male) for DTI: -0.037


In [38]:
# 8.6.2 DTI ↔ Approval (approval rate by DTI quantiles)

dti_out = df[["y", "fin_debt_to_income"]].dropna().copy()
dti_out["y"] = dti_out["y"].astype(int)

dti_out["dti_bin"] = pd.qcut(dti_out["fin_debt_to_income"], q=5, duplicates="drop")
dti_bin_stats = dti_out.groupby("dti_bin", observed=True)["y"].agg(n="count", approval_rate="mean")
display(dti_bin_stats)

corr_dti = dti_out[["fin_debt_to_income", "y"]].corr().iloc[0, 1]
print(f"Correlation(DTI, approval y): {corr_dti:.3f}")

,n,approval_rate
dti_bin,,
"(0.049, 0.12]",103,0.631068
"(0.12, 0.2]",111,0.558559
"(0.2, 0.28]",95,0.526316
"(0.28, 0.37]",102,0.607843
"(0.37, 1.85]",91,0.582418


Correlation(DTI, approval y): 0.007


**Conclusion (8.6 — Debt-to-income ratio as a proxy candidate)**  

**Link (1): DTI ↔ Gender.** DTI distributions are nearly identical across gender groups (female mean ≈ 0.244 vs male mean ≈ 0.249). The standardized mean difference is **SMD = -0.037**, indicating a **negligible** gender difference. This suggests **no meaningful proxy potential** for gender via DTI.

**Link (2): DTI ↔ Approval.** Approval rates across DTI quantiles do not show a consistent monotonic pattern, and the correlation between DTI and approval is essentially zero (**r = 0.007**). This suggests DTI is **not strongly decision-relevant** in this dataset.

**Overall:** DTI is neither a strong gender proxy nor a strong predictor of approval outcomes based on the current evidence.

## 8.7 Proxy risk summary (ZIP + financial variables)

We evaluated proxy risk using a two-link logic: a feature is more concerning when it is associated with **(i) gender** and also with **(ii) approval outcomes**.

- **ZIP code:** ZIP shows proxy potential for gender (ZIP ↔ gender appears significant), but the ZIP×gender table is extremely sparse (χ² assumptions violated). After grouping ZIPs (top 20 + OTHER), we still do **not** find strong evidence that ZIP is associated with approval outcomes (p = 0.166), so a robust proxy-to-outcome pathway is **not confirmed**.

- **Annual income:** Income shows **only a small gender difference** (SMD ≈ 0.10), but is **clearly decision-relevant** (approval rates increase substantially across income quantiles; corr ≈ 0.178). This suggests income is likely acting as a **risk-related driver**, rather than a strong gender proxy.

- **Debt-to-income (DTI):** DTI shows **negligible gender difference** (SMD ≈ -0.04) and **near-zero association** with approval (corr ≈ 0.007), indicating low proxy and low decision relevance in this dataset.

**Overall:** The strongest confirmed driver among tested candidates is **income** (outcome link), while evidence for proxy discrimination via ZIP or DTI is limited under the current tests. These findings support focusing mitigation on outcome monitoring (DI) and careful governance of features with demographic encoding potential (e.g., geographic granularity).

## 9. Limitations

- DI and statistical tests are descriptive and do not prove causality or intent.
- Dataset size and simulated artifacts may affect estimates.
- Unobserved confounders can explain some observed disparities.

## 10. Recommendations 

We propose actionable controls such as:
- monitoring DI over time with alerts (e.g., DI < 0.8)
- reviewing or justifying strong proxy variables (e.g., ZIP code)
- documentation of feature usage and decision rationale
- human oversight and audit trails for high-risk decisions

## 11. Key Findings Summary (for README / Video)

We summarize:
- group sizes and selection rates
- DI value and whether the 80% rule is triggered
- strongest proxy risk signals identified